In [1]:
import mlflow
# Step 1: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://184.72.71.39:5000/")

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("LightGBM HP Tuning")

2026/09/09 18:47:35 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://comment-analysis-bucket-994/7', creation_time=1788976055197, effective_trace_archival_retention=None, experiment_id='7', last_update_time=1788976055197, lifecycle_stage='active', name='LightGBM HP Tuning', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('../data/processed/processed_comments.csv').dropna()
df.shape

(36662, 2)

In [5]:
# Final LightGBM tuning experiment

df = df.dropna(subset=['category', 'clean_comment']).copy()

# Safe even if labels were already changed in an earlier cell
df['category'] = df['category'].replace({-1: 2})

ngram_range = (1, 3)
max_features = 1000


# Final train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)


# Validation split for Optuna
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)


# TF-IDF fitted only on inner training data
vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_inner_vec = vectorizer.fit_transform(X_train_inner)
X_val_vec = vectorizer.transform(X_val)


# SMOTE only on inner training data
smote = SMOTE(random_state=42)

X_train_inner_vec, y_train_inner = smote.fit_resample(
    X_train_inner_vec,
    y_train_inner
)


def objective_lightgbm(trial):

    params = {
        'n_estimators': trial.suggest_int(
            'n_estimators', 100, 1000
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate', 1e-4, 1e-1, log=True
        ),

        'max_depth': trial.suggest_int(
            'max_depth', 3, 15
        ),

        'num_leaves': trial.suggest_int(
            'num_leaves', 20, 150
        ),

        'min_child_samples': trial.suggest_int(
            'min_child_samples', 10, 100
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree', 0.5, 1.0
        ),

        'subsample': trial.suggest_float(
            'subsample', 0.5, 1.0
        ),

        'reg_alpha': trial.suggest_float(
            'reg_alpha', 1e-4, 10.0, log=True
        ),

        'reg_lambda': trial.suggest_float(
            'reg_lambda', 1e-4, 10.0, log=True
        )
    }

    model = LGBMClassifier(
        **params,
        random_state=42,
        subsample_freq=1,
        verbosity=-1,
        n_jobs=-1
    )

    model.fit(
        X_train_inner_vec,
        y_train_inner
    )

    y_pred = model.predict(X_val_vec)

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    # Log each Optuna trial as a separate MLflow run
    with mlflow.start_run(
        run_name=f"LightGBM_Trial_{trial.number}"
    ):

        mlflow.set_tag(
            "experiment_type",
            "final_lightgbm_tuning"
        )

        mlflow.log_param(
            "algo_name",
            "LightGBM"
        )

        mlflow.log_params(params)

        mlflow.log_metric(
            "validation_accuracy",
            accuracy
        )

    return accuracy


# Optuna
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective_lightgbm,
    n_trials=50
)


best_params = study.best_params

print("Best parameters:")
print(best_params)

print(
    "Best validation accuracy:",
    study.best_value
)


# ----------------------------
# Train final winning model
# ----------------------------

# Refit TF-IDF using ALL training data
final_vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)


# SMOTE only on final training data
final_smote = SMOTE(
    random_state=42
)

X_train_resampled, y_train_resampled = final_smote.fit_resample(
    X_train_vec,
    y_train
)


best_model = LGBMClassifier(
    **best_params,
    random_state=42,
    subsample_freq=1,
    verbosity=-1,
    n_jobs=-1
)

best_model.fit(
    X_train_resampled,
    y_train_resampled
)


# Final evaluation on untouched test set
y_pred = best_model.predict(
    X_test_vec
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "Final test accuracy:",
    accuracy
)


classification_rep = classification_report(
    y_test,
    y_pred,
    output_dict=True
)


# ----------------------------
# Log final model
# ----------------------------

with mlflow.start_run(
    run_name="Best_LightGBM_Final_Model"
):

    mlflow.set_tag(
        "experiment_type",
        "final_lightgbm_model"
    )

    mlflow.log_param(
        "algo_name",
        "LightGBM"
    )

    mlflow.log_param(
        "vectorizer_type",
        "TF-IDF"
    )

    mlflow.log_param(
        "ngram_range",
        str(ngram_range)
    )

    mlflow.log_param(
        "max_features",
        max_features
    )

    mlflow.log_param(
        "imbalance_method",
        "SMOTE"
    )

    mlflow.log_param(
        "optuna_trials",
        50
    )

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    for label, metrics in classification_rep.items():

        if isinstance(metrics, dict):

            for metric, value in metrics.items():

                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    # Correct flavor for LightGBM
    mlflow.lightgbm.log_model(
        best_model,
        name="LightGBM_model"
    )


print("Final LightGBM experiment completed.")


# Optional Optuna visualizations
# Wrapped so a plotting problem cannot ruin the experiment
try:
    optuna.visualization.plot_param_importances(
        study
    ).show()

    optuna.visualization.plot_optimization_history(
        study
    ).show()

except Exception as e:
    print("Optuna plots could not be displayed:", e)

[I 2026-09-09 19:00:13,042] A new study created in memory with name: no-name-9561163a-6816-4eb0-8420-3e6a888ea18e


🏃 View run LightGBM_Trial_0 at: http://184.72.71.39:5000/#/experiments/7/runs/73c4c8fcb42544c3ac8f3f4b59ff8d56
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/7


[I 2026-09-09 19:00:30,344] Trial 0 finished with value: 0.6537674735765427 and parameters: {'n_estimators': 303, 'learning_rate': 0.0003200502219419558, 'max_depth': 12, 'num_leaves': 140, 'min_child_samples': 90, 'colsample_bytree': 0.8907163284042957, 'subsample': 0.9378678871280774, 'reg_alpha': 2.841012514010223, 'reg_lambda': 2.188359511449835}. Best is trial 0 with value: 0.6537674735765427.


🏃 View run LightGBM_Trial_1 at: http://184.72.71.39:5000/#/experiments/7/runs/63b0d1dcba4743e3a6158826965f4f35
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/7


[I 2026-09-09 19:00:37,870] Trial 1 finished with value: 0.6694510739856802 and parameters: {'n_estimators': 112, 'learning_rate': 0.0010056263185846681, 'max_depth': 12, 'num_leaves': 114, 'min_child_samples': 92, 'colsample_bytree': 0.6824798327120478, 'subsample': 0.5134596418664534, 'reg_alpha': 0.0002118090132853456, 'reg_lambda': 0.00011234910623501516}. Best is trial 1 with value: 0.6694510739856802.


...

🏃 View run LightGBM_Trial_48 at: http://184.72.71.39:5000/#/experiments/7/runs/63c748fc607347e381cfbca1e5009a4a
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/7


[I 2026-09-09 19:15:44,673] Trial 48 finished with value: 0.7865666553017389 and parameters: {'n_estimators': 936, 'learning_rate': 0.03407359989787098, 'max_depth': 11, 'num_leaves': 122, 'min_child_samples': 71, 'colsample_bytree': 0.5397148054019762, 'subsample': 0.9159587311799324, 'reg_alpha': 0.07472926776734093, 'reg_lambda': 0.39109160521433517}. Best is trial 38 with value: 0.7891237640640982.


🏃 View run LightGBM_Trial_49 at: http://184.72.71.39:5000/#/experiments/7/runs/8fe93d0a5c1a4d068e2e17dec5bfce32
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/7


[I 2026-09-09 19:16:06,475] Trial 49 finished with value: 0.7884418683941357 and parameters: {'n_estimators': 707, 'learning_rate': 0.032354366761872, 'max_depth': 12, 'num_leaves': 135, 'min_child_samples': 59, 'colsample_bytree': 0.5167290155151603, 'subsample': 0.9259190219921806, 'reg_alpha': 0.018015522346816408, 'reg_lambda': 0.24411715380801077}. Best is trial 38 with value: 0.7891237640640982.


Best parameters:
{'n_estimators': 676, 'learning_rate': 0.03469871112689436, 'max_depth': 12, 'num_leaves': 141, 'min_child_samples': 74, 'colsample_bytree': 0.5018738817017754, 'subsample': 0.9244835903080109, 'reg_alpha': 0.021460517234316315, 'reg_lambda': 0.8490115177337595}
Best validation accuracy: 0.7891237640640982
Final test accuracy: 0.785762989226783
🏃 View run Best_LightGBM_Final_Model at: http://184.72.71.39:5000/#/experiments/7/runs/94467f81c25d463abd6e32426dd34fe9
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/7
Final LightGBM experiment completed.
Optuna plots could not be displayed: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.
